## NeuroMTA Framework Example
### Example 3: Multi-threading

In [1]:
import torch
from neuromta.framework import *

In [ ]:
class SimpleNPUCore(Core):
    def __init__(self, core_id):
        super().__init__(core_id, SimpleNPUCoreCycleModel())
    
        self.l1_memory = MemoryHandle(
            mem_id="L1", 
            base_addr=0x00, 
            size=parse_mem_cap_str("2MB")
        )
        
        self.mxu_pe_arr = torch.zeros((128, 128), dtype=torch.int32)
        
        self.mxu_lock = self.l1_memory.allocate_var_ptr(4, initial_value=0)
        self.l1_read_lock = self.l1_memory.allocate_var_ptr(4, initial_value=0)
        self.l1_write_lock = self.l1_memory.allocate_var_ptr(4, initial_value=0)
    
    @core_command_method
    def mxu_compute(
        self, 
        
        ifm: DataContainer[torch.Tensor], 
        wgt: DataContainer[torch.Tensor], 
        psum: DataContainer[torch.Tensor], 
        ofm: DataContainer[torch.Tensor],
    
        preload_psum: bool = True,
        flush_ofm: bool = True,
    ):
        if preload_psum:
            psum.data = psum.data.view(torch.int32).reshape(128, 128)
            self.mxu_pe_arr[:, :] = psum.data

        ifm.data = ifm.data.view(torch.int32).reshape(128, 128)
        wgt.data = wgt.data.view(torch.int32).reshape(128, 128)
        
        self.mxu_pe_arr[:, :] = torch.matmul(ifm.data, wgt.data) + self.mxu_pe_arr

        if flush_ofm:
            ofm.data = self.mxu_pe_arr.clone()
            self.mxu_pe_arr[:, :] = 0
        
    @core_command_method
    def l1_read_single_page(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        container.data = self.l1_memory.get_content(ptr, shape=(128, 128), dtype=torch.int32)

    @core_command_method
    def l1_write_single_page(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        self.l1_memory.set_content(ptr, container.data)
        
    @core_conditional_command_method
    def lock_wait(self, lock_ptr: Pointer, value: int):
        return self.l1_memory.get_content(lock_ptr) == value
    
    @core_command_method
    def lock_atomic_inc(self, lock_ptr: Pointer):
        current_value = self.l1_memory.get_content(lock_ptr)
        self.l1_memory.set_content(lock_ptr, current_value + 1)

class SimpleNPUCoreCycleModel(CoreCycleModel):
    def __init__(self):
        super().__init__()
        
    def mxu_compute(
        self,
        ifm: DataContainer[torch.Tensor],
        wgt: DataContainer[torch.Tensor],
        psum: DataContainer[torch.Tensor],
        ofm: DataContainer[torch.Tensor],
        preload_psum: bool = True,
        flush_ofm: bool = True,
    ):
        i = 1
        if preload_psum: i += 1
        if flush_ofm: i += 1
        return 128 * i 
    
    def l1_read_single_page(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        return 64
    
    def l1_write_single_page(self, ptr: BufferPointer, container: DataContainer[torch.Tensor]):
        return 64

In [3]:
class SimpleNPUDevice(Device):
    def __init__(self, n_cores: int):
        super().__init__()
        
        self.npu_cores = [SimpleNPUCore(core_id=i) for i in range(n_cores)]

In [ ]:
@jit_prototype
def example_kernel(
    core: SimpleNPUCore, 
    
    ifm: BufferPointer,
    wgt: BufferPointer,
    psum: BufferPointer,
    ofm: BufferPointer,
    
    n_reqs: int = 4,
):
    for i in range(n_reqs):
        with new_parallel_thread():  # create a new parallel thread
            containers = [DataContainer() for _ in range(4)]
            
            core.lock_wait(core.l1_read_lock, i)                # CRITICAL SECTION START
            core.l1_read_single_page(ifm[i], containers[0])     #   load input feature map into the data container
            core.l1_read_single_page(wgt[i], containers[1])     #   load weight into the data container
            core.l1_read_single_page(psum[i], containers[2])    #   load partial sum into the data container
            core.lock_atomic_inc(core.l1_read_lock)             # CRITICAL SECTION END
            
            core.lock_wait(core.mxu_lock, i)                    # CRITICAL SECTION START
            core.mxu_compute(*containers)                       #   perform matrix multiplication and accumulation
            core.lock_atomic_inc(core.mxu_lock)                 # CRITICAL SECTION END
            
            core.lock_wait(core.l1_write_lock, i)               # CRITICAL SECTION START
            core.l1_write_single_page(ofm[i], containers[3])    #   store output feature map from the data container
            core.lock_atomic_inc(core.l1_write_lock)            # CRITICAL SECTION END

    core.parallel_merge()  # optional: wait for all threads to complete

In [5]:
device = SimpleNPUDevice(n_cores=1)  # Initialize device with 1 core
device.initialize()

In [6]:
logger.set_print_options(log_level=LogLevel.DEBUG)
device.set_command_debug_verbosity(verbose=True)

In [7]:
core = device.npu_cores[0]
n_reqs = 4

ifm  = core.l1_memory.allocate_buffer_ptr(page_size=128*128*4, n_pages=n_reqs)
wgt  = core.l1_memory.allocate_buffer_ptr(page_size=128*128*4, n_pages=n_reqs)
psum = core.l1_memory.allocate_buffer_ptr(page_size=128*128*4, n_pages=n_reqs)
ofm  = core.l1_memory.allocate_buffer_ptr(page_size=128*128*4, n_pages=n_reqs)

ifm_tensor  = torch.randint(0, 10, (4, 128, 128), dtype=torch.int32)
wgt_tensor  = torch.randint(0, 10, (4, 128, 128), dtype=torch.int32)
psum_tensor = torch.randint(0, 10, (4, 128, 128), dtype=torch.int32)

core.l1_memory.set_content(ifm,  ifm_tensor)
core.l1_memory.set_content(wgt,  wgt_tensor)
core.l1_memory.set_content(psum, psum_tensor)

In [8]:
kernel = example_kernel(core, ifm, wgt, psum, ofm, n_reqs=n_reqs)
kernel.dispatch(slot_id="main")

device.run_kernels()

[DEBUG] 0      - 1      | 0          | MAIN<main>::example_kernel::0                                                                        | command: lock_wait
[DEBUG] 1      - 65     | 0          | MAIN<main>::example_kernel::0                                                                        | command: l1_read_single_page
[DEBUG] 65     - 129    | 0          | MAIN<main>::example_kernel::0                                                                        | command: l1_read_single_page
[DEBUG] 129    - 193    | 0          | MAIN<main>::example_kernel::0                                                                        | command: l1_read_single_page
[DEBUG] 193    - 194    | 0          | MAIN<main>::example_kernel::0                                                                        | command: lock_atomic_inc
[DEBUG] 0      - 194    | 0          | MAIN<main>::example_kernel::1                                                                        | command: lock_wai

In [9]:
print(f"simulation terminated in {core.timestamp} cycles")

for i in range(n_reqs):
    reference = torch.matmul(ifm_tensor[i], wgt_tensor[i]) + psum_tensor[i]
    simulated = core.l1_memory.get_content(ofm[i], shape=(128, 128), dtype=torch.int32)
    print(f"simulation {'PASSED' if torch.equal(reference, simulated) else 'FAILED'}")

simulation terminated in 1865 cycles
simulation PASSED
simulation PASSED
simulation PASSED
simulation PASSED
